# HUDM 5001 — Programming for Data Science
## Session 02 · Python Intro: Operators, Input/Output & NumPy
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youmilab/hudm5001/blob/main/colab_notebooks/02_Operators_IO_Numpy.ipynb)

[**Chenguang Pan**](https://cpan.ai/), [**Youmi Lab**](https://youmilab.ai/)  
**Teachers College, Columbia University**   
  
This notebook is used in a mini-session designed to help interested learners quickly grasp fundamental Python skills. For students of HUDM5001 in Fall 2026 semester, please look for [this repo](https://github.com/cgpan/hudm5001) for class materials.  
  
This notebook is based on Prof. Youmi Suk's original design for Programming for Data Science. Original materials can be found [here](https://github.com/youmilab/hudm5001).

> ▶️ **How to open this notebook in Google Colab** — pick whichever is easiest:
> 1. **From Google Drive:** put this `.ipynb` in your Drive, right-click it → *Open with* → *Google Colaboratory*.
> 2. **Upload:** go to [colab.research.google.com](https://colab.research.google.com) → *File ▸ Upload notebook* → choose this file.
> 3. **From GitHub:** click the **Open in Colab** badge above (works once the notebook is pushed to the course repo).
>
> Then **run the Setup cell first** (▶ or `Shift+Enter`). Everything else runs top-to-bottom.

Three building blocks today. **Operators** let us *compute* with values. **Input/Output** lets us *read and write* files (text, JSON, CSV). And **NumPy** gives us the fast numeric arrays that all of scientific Python is built on.

---
## 1. Big Picture

### 🎯 Learning Objectives
By the end of this session you will be able to:
- Use arithmetic (`+ - * / // % **`), comparison, logical, membership, and identity operators.
- Read and write **text**, **JSON**, and **CSV** files using `open()` / `with`.
- Manage paths safely with the `os` / `os.path` module.
- Create **NumPy `ndarray`s** and understand *vectorized*, element-wise operations.
- Index, slice, and **boolean-index** arrays; compute summary statistics.

### ✅ Prerequisites
- Week 3 (data types, variables).
- Run the Setup cell to import numpy & pandas.

### 🧩 Key Concepts
`arithmetic`  `comparison`  `logical`  `identity (`is`)`  `membership (`in`)`  `file modes`  `JSON`  `CSV`  `ndarray`  `vectorization`  `boolean indexing`

### 📖 Reading
- McKinney, *Python for Data Analysis*, Chapter 4 (NumPy basics)

### 🗺️ Today's Agenda
1. Operators: arithmetic, comparison, logical, identity, membership
2. Input/Output: text, JSON, CSV files; `with open(...)`
3. Paths with `os.path`
4. NumPy motivation & the ndarray
5. Array creation, math, indexing, slicing, boolean indexing

---
## ▶️ Setup — run this cell first
This installs anything Colab is missing, imports the libraries we'll use, and sets `DATA_BASE`
(the address of the course data on GitHub).

In [ ]:
# === SETUP — run me first! ===
import numpy as np
import pandas as pd
import json, os

# Course data is read straight from the public course repo on GitHub,
# so these notebooks run anywhere (Colab, Jupyter, locally) with no downloads.
DATA_BASE = "https://raw.githubusercontent.com/youmilab/hudm5001/main/"
print("Setup complete ✅  — data base URL:", DATA_BASE)

---
## 2. Content

### 2.1 Operators
**Arithmetic.** Beyond `+ - * /`, note three you may not know:
- `//` **floor division** (divide and round *down*),
- `%` **modulus** (the remainder),
- `**` **exponentiation**.

In [ ]:
print(5 / 2)    # 2.5  (true division -> always float)
print(5 // 2)   # 2    (floor division)
print(-5 // 2)  # -3   (rounds DOWN, toward negative infinity)
print(5 % 2)    # 1    (remainder -> great for 'is it even/odd?')
print(5 ** 3)   # 125  (5 cubed)

**Comparison** operators return a `bool`: `==  !=  <  <=  >  >=`.
**Logical** operators combine bools: `and  or  not`.

In [ ]:
x = 10
print(0 == (10 % 5))            # True  (10 % 5 == 0)
print(5/9 != 0.5555)           # True
print((x % 10 == 0) and (x < -1))   # False
print(not x == 5)              # True

**Identity** `is` / `is not` asks *"the same object in memory?"* — different from `==`
(*"the same value?"*). **Membership** `in` / `not in` tests if a value is inside a container.

In [ ]:
x = ['python', 'R']
y = ['python', 'R']     # same contents, different object
print(x == y)           # True  -> equal values
print(x is y)           # False -> NOT the same object

print('fries' in ['fries', 'burger'])     # True
print('chicken' not in ['fries', 'burger'])  # True

#### ✏️ Try It Yourself — Exercise 1
Use `%` to print every number from 1 to 20 that is divisible by 3. (Loop with `range`, test `n % 3 == 0`.)

In [ ]:
# Your code here

<details>
<summary>▶ Show solution</summary>

```python
for n in range(1, 21):
    if n % 3 == 0:
        print(n, end=' ')
```

</details>

### 2.2 Input/Output — reading & writing files
You open a file with `open(path, mode)`. Common **modes**: `'r'` read, `'w'` write (overwrite), `'a'` append.
Always prefer the **`with open(...)`** form — it closes the file automatically, even if an error occurs.

**Text files.** Let's write one, then read it back (this runs in your Colab session).

In [ ]:
# WRITE
with open('notes.txt', 'w') as f:
    f.write('line 1: hello\n')
    f.write('line 2: programming for data science\n')

# READ
with open('notes.txt', 'r') as f:
    contents = f.read()
print(contents)

**JSON** (JavaScript Object Notation) is text that maps cleanly to Python dicts/lists — the most
common format for web data. Use `json.dump` to write and `json.load` to read.

In [ ]:
record = {'name': 'iris study', 'n': 150, 'features': ['sepal', 'petal']}

with open('record.json', 'w') as f:
    json.dump(record, f)

with open('record.json', 'r') as f:
    back = json.load(f)
print(back)
print('features:', back['features'])      # back is a normal dict

**CSV** (comma-separated values) is the everyday data format. We'll read one **straight from the
course GitHub repo** via `DATA_BASE` — no download needed — using pandas (much easier than the raw `csv` module).

In [ ]:
wine = pd.read_csv(DATA_BASE + 'lecture_notes/python/data_example_wine.csv')
print(type(wine), '| shape:', wine.shape)
print('columns:', list(wine.columns))
wine.head()

In [ ]:
# WRITE a CSV: keep just the first two rows and save locally
wine.head(2).to_csv('wine_first_two.csv', index=False)
print(open('wine_first_two.csv').read())

**Paths with `os.path`.** Build and inspect paths portably (works on Mac/Windows/Linux).

In [ ]:
print('cwd:', os.getcwd())

some_path = '/Users/clark_kent/data.csv'
print('basename:', os.path.basename(some_path))   # 'data.csv'
print('dirname :', os.path.dirname(some_path))     # '/Users/clark_kent'
print('joined  :', os.path.join('data', 'raw', 'file.csv'))   # OS-correct separator
print('exists? :', os.path.exists('notes.txt'))    # True (we wrote it above)

#### ✏️ Try It Yourself — Exercise 2
Read the wine CSV from `DATA_BASE` again, then write **only its column names** to a text file called `wine_cols.txt` (one per line) and print the file back.

In [ ]:
# Your code here

<details>
<summary>▶ Show solution</summary>

```python
wine = pd.read_csv(DATA_BASE + 'lecture_notes/python/data_example_wine.csv')
with open('wine_cols.txt', 'w') as f:
    for col in wine.columns:
        f.write(col + '\n')
print(open('wine_cols.txt').read())
```

</details>

### 2.3 NumPy — fast numeric arrays
**Why NumPy?** A Python list of a million numbers is slow to do math on. NumPy stores numbers in a compact
**`ndarray`** and applies operations to the *whole array at once* (**vectorization**) — far faster and cleaner
than looping.

In [ ]:
# A quick taste: 10 random dice rolls in [1, 6]
np.random.seed(0)                    # reproducible randomness
print(np.random.randint(1, 7, 10))

An **ndarray** is a multidimensional, *homogeneous* (all-one-type), fixed-size grid. Make one from
a list of lists; `.shape` gives `(rows, columns)`.

In [ ]:
x = np.array([[2, 3, 1],
              [3, 1, 2]])
print(x)
print('shape:', x.shape)     # (2, 3) -> 2 rows, 3 cols
print('dtype:', x.dtype)

**Vectorized math** happens *element-wise* — no loop required.

In [ ]:
print(x * 2)      # scale every element
print('---')
print(x + x)      # element-wise addition
print('---')
print(1 / x)      # element-wise reciprocal

**Special arrays** you'll reach for constantly:

In [ ]:
print(np.zeros((2, 3)))     # all zeros, shape 2x3
print(np.ones((2, 3)))      # all ones
print(np.identity(3))       # 3x3 identity matrix
print(np.arange(0, 10, 2))  # like range, but an array: [0 2 4 6 8]

**Indexing & slicing** works like lists, extended to multiple dimensions with `[row, col]`.
**Boolean indexing** selects elements that satisfy a condition — the key to filtering data.

In [ ]:
np.random.seed(1)
z = np.random.randn(5)          # 5 draws from a standard normal
print('z        :', z.round(2))
print('z[1:3]   :', z[1:3].round(2))      # slice positions 1,2
print('z > 0    :', z > 0)                # a boolean array
print('positives:', z[z > 0].round(2))    # boolean indexing -> keep only z>0

In [ ]:
# 2-D indexing: rows then columns
m = np.array([[10, 20, 30],
              [40, 50, 60]])
print('element [0,2]:', m[0, 2])      # 30
print('first row    :', m[0, :])      # [10 20 30]
print('last column  :', m[:, -1])     # [30 60]

# common summaries
print('mean:', m.mean(), '| max:', m.max(), '| column sums:', m.sum(axis=0))

#### ✏️ Try It Yourself — Exercise 3
Make an array `w = np.arange(1, 11)` (the numbers 1–10). Using **boolean indexing**, print (a) only the even numbers and (b) the mean of the numbers greater than 5.

In [ ]:
# Your code here

<details>
<summary>▶ Show solution</summary>

```python
w = np.arange(1, 11)
print(w[w % 2 == 0])           # even numbers
print(w[w > 5].mean())        # mean of those > 5
```

</details>

---
## 3. Conclusions & Key Takeaways
- `//` floors, `%` gives the remainder, `**` is power; `/` always returns a float.
- `is` compares **identity** (same object); `==` compares **value**. They're not the same!
- Always read/write files with **`with open(path, mode)`** so they close safely.
- **JSON ↔ dict** via `json.load`/`json.dump`; **CSV** is easiest with `pd.read_csv` / `df.to_csv`.
- You can read data **directly from a URL** with pandas — that's how these notebooks stay zero-setup.
- NumPy's **ndarray** enables **vectorized** element-wise math — fast and loop-free.
- **Boolean indexing** (`arr[arr > 0]`) is the foundation of data filtering in NumPy and pandas.

### 🧾 Quick Reference
| Task | Code |
|---|---|
| floor / remainder | `a // b`, `a % b` |
| read text safely | `with open(p) as f: f.read()` |
| read CSV from URL | `pd.read_csv(DATA_BASE + '...')` |
| array from list | `np.array([[1,2],[3,4]])` |
| zeros / ones / identity | `np.zeros((r,c))`, `np.ones(...)`, `np.identity(n)` |
| element-wise math | `x * 2`, `x + y`, `1 / x` |
| boolean filter | `z[z > 0]` |
| stats | `arr.mean()`, `.sum(axis=0)`, `.max()` |

### ⏭️ Coming Up Next
**Week 5 — pandas!** The library you'll use most: labeled tables (DataFrames) built on top of NumPy. Read McKinney Ch 5.

### 📌 This Week's Assignment
NumPy programming assignment (e.g., the stock-price / dice exercises) — see GitHub.

> 💡 **Tip:** make a copy in Colab (*File ▸ Save a copy in Drive*) and add your own notes — experimenting in the code cells is the fastest way to learn.